# Lecture 3: Data Retrieval

:::{admonition} Learning Objectives
:class: tip
After this lecture, you will be able to:
- Explain why efficient data retrieval matters at scale
- Choose appropriate data formats (CSV vs Parquet) and data types
- Write performant queries using Polars and DuckDB
- Understand when to use in-memory processing vs databases
- Interact with relational databases from Python
- Navigate the data storage and management landscape {cite}`kleppmann2017designing`
- Explain the difference between row-oriented and columnar storage formats
- Recognise when a dataset has outgrown pandas and identify the right next tool
:::

This lecture builds on the pandas material from [Lecture 2](lecture_2.ipynb) and prepares the ground for [Lecture 4](lecture_4.ipynb), where we move from data engineering into modelling. The storage and database concepts here follow the systems perspective of {cite}`kleppmann2017designing`; the "know your regime" advice draws on {cite}`huyen2022designing`.

## Table of Contents

- [Motivation: working with (biggish) data](#motivation-working-with-biggish-data)
- [How to speed things up](#how-to-speed-things-up)
- [Python and databases](#python-and-databases)
- [Data storage & management landscape](#data-storage-management-landscape)
- [Repository structure: .gitignore, pre-commit, type hints](#repository-structure-gitignore-pre-commit-type-hints)
- [Scale awareness: when pandas is not enough](#scale-awareness-when-pandas-is-not-enough)

A [Key Takeaways](#key-takeaways) box and an [Exercises](#exercises) section close the lecture.

:::{note}
This notebook imports from the `fun_ds` package. To run the cells, clone the
repository and follow the setup instructions on the [home page](/intro).
:::

In [ ]:
import os
import tempfile
import time
import pathlib

import numpy as np
import pandas as pd

from fun_ds.data import load_california_housing
from fun_ds.plotting import set_lecture_style

set_lecture_style()
pd.set_option("display.max_columns", 50)

df = load_california_housing()
print(df.shape)
df.head()

## Motivation: Working with (Biggish) Data

Data science is, before anything else, a *data-first* discipline. Models — however sophisticated — inherit the quality, coverage, and structure of the data they are fitted on, and the practitioner's day-to-day time budget reflects that. A widely cited figure among practitioners is that data scientists spend roughly **60–80% of their time** on data collection, cleaning, and preparation rather than on modelling itself {cite}`kuhn2019feature,huyen2022designing`. Whether the exact number is 60 or 80, the qualitative point is uncontroversial: fluency with retrieval and storage is not a peripheral skill but the *central* one.

If you do not use the right tools, you will spend most of your time **waiting** on data.

Two motivating case studies:
- Automating a business process for an insurance company (>20 tables, >500M rows, >300 columns)
- Training an LLM (Llama 3.1: 15 trillion tokens of pre-training data)

$\rightarrow$ Being able to handle large amounts of data efficiently is a key skill, and — as {cite}`huyen2022designing` argues for production ML — often the difference between a system that ships and one that stalls in a notebook.

## How to Speed Things Up

We will load >178 million rows in <2 seconds. The key principles:
1. Only use what you need
2. Choose the right data types
3. Optimise your queries
4. Understand in-memory vs on-disk trade-offs
5. Use appropriate database management systems

### Why You Should Not Work with CSV

:::{warning}
CSV files are human-readable but problematic at scale:
- No type information (everything is text)
- No compression
- Must read entire file to query a subset
- Parsing is slow and error-prone
:::

#### Row- vs. column-oriented storage {cite}`kleppmann2017designing`

The deeper reason CSV struggles at analytical scale is that it is **row-oriented**: bytes on disk follow the order *(row 1, all columns) → (row 2, all columns) → …*. This is the natural layout for **OLTP** (Online Transaction Processing) workloads — think of an insurance claims system where each transaction touches a small number of complete records. In contrast, **columnar** formats such as Apache Parquet lay bytes out in *column chunks*: all values of column A, then all values of column B, and so on. This is the natural layout for **OLAP** (Online Analytical Processing) workloads — aggregations, filters, and joins over a handful of columns across many rows {cite}`kleppmann2017designing`.

Two properties follow directly from the columnar layout, and together they explain almost every reported performance advantage of Parquet over CSV under typical analytical workloads:

1. **Higher compressibility.** Values *within* a column chunk tend to be similar in type and often in magnitude (e.g. all `float64` incomes, all short country codes). Encoding schemes such as run-length encoding, dictionary encoding, and bit-packing exploit this local homogeneity and routinely deliver 5–10× on-disk compression. A row-oriented layout mixes types byte-by-byte and cannot compress as effectively.
2. **Less I/O per query.** Analytical queries usually touch a small subset of columns (`SELECT MedInc, MedHouseVal ...`). With a column layout, the query engine can perform *projection pushdown* and read only the relevant column chunks from disk. With CSV every query pays for reading every column.

:::{note}
Both properties are qualitative; the exact speed-up depends on the compression codec, the query, the storage medium, and the CPU. Claims like "Parquet is 10× faster" should therefore always be read as "under typical analytical workloads, on this dataset, with this engine".
:::

In [ ]:
# CSV vs Parquet: file size and load time on ~100k rows of California Housing.
# We repeat the housing frame to reach ~100k rows and write to a temp dir.

rows = pd.concat([df] * 5, ignore_index=True)   # ~103k rows
print(f"benchmark frame shape: {rows.shape}")

with tempfile.TemporaryDirectory() as tmp:
    csv_path = pathlib.Path(tmp) / "housing.csv"
    parquet_path = pathlib.Path(tmp) / "housing.parquet"

    rows.to_csv(csv_path, index=False)
    try:
        rows.to_parquet(parquet_path, index=False)
        parquet_available = True
    except ImportError:
        parquet_available = False
        print("pyarrow/fastparquet not installed; install with `pip install pyarrow` to run the Parquet half.")

    csv_size = csv_path.stat().st_size / 1e6
    print(f"CSV file size:     {csv_size:6.2f} MB")

    tic = time.time()
    _ = pd.read_csv(csv_path)
    csv_load = time.time() - tic
    print(f"CSV load time:     {csv_load:6.3f} s")

    if parquet_available:
        parquet_size = parquet_path.stat().st_size / 1e6
        print(f"Parquet file size: {parquet_size:6.2f} MB  ({csv_size / parquet_size:.1f}x smaller)")

        tic = time.time()
        _ = pd.read_parquet(parquet_path)
        parquet_load = time.time() - tic
        print(f"Parquet load time: {parquet_load:6.3f} s  ({csv_load / parquet_load:.1f}x faster)")

**Expected output.** On typical hardware, the Parquet file is roughly 5–10× smaller than the CSV and loads several times faster because it stores each column contiguously with per-column compression. The gap widens as datasets grow.

#### Schema evolution

Real datasets rarely have a fixed schema over their lifetime: business logic changes, new features are collected, deprecated fields are dropped. **Schema evolution** is the study of how storage formats accommodate these changes without full rewrites.

Parquet handles schema evolution *asymmetrically*:

- **Adding a column is cheap.** New files are written with the extra column; older files without it are read as `NULL` for that column. This is because Parquet stores each column as an independent chunk with its own metadata.
- **Removing a column is cheap.** Readers simply project the columns they still need.
- **Renaming a column is cheap** in most engines (metadata-only), but subtle: some readers key on column name, others on ordinal position.
- **Changing a column's type is expensive.** Column chunks are physically encoded for a specific logical type; converting `int32` → `int64` or `string` → `categorical` requires rewriting the affected files.

The take-away for pipeline design: prefer **additive** schema changes; treat type changes as a versioned migration, not a silent edit {cite}`kleppmann2017designing`.

:::{note}
**Apache Arrow — the in-memory columnar standard.** Parquet is a *disk* format. Its in-memory sibling is [Apache Arrow](https://arrow.apache.org/), a language-independent columnar memory layout that lets pandas, Polars, DuckDB, and Spark exchange data *without copying or serialising*. When you read a Parquet file with pandas 2.x or Polars, the bytes are decoded straight into Arrow buffers that any Arrow-aware engine can consume. This unification is what makes it practical to move a table from a DuckDB query into a Polars pipeline into a scikit-learn matrix without paying the usual serialisation cost at every hop.
:::

### Only Use What You Need

Column projection and row filtering at read time dramatically reduce I/O.

In [ ]:
# Column projection: read only the columns you need from Parquet.

with tempfile.TemporaryDirectory() as tmp:
    parquet_path = pathlib.Path(tmp) / "housing.parquet"

    try:
        df.to_parquet(parquet_path, index=False)
    except ImportError:
        print("Install `pyarrow` to run this demo (`pip install pyarrow`).")
    else:
        tic = time.time()
        full = pd.read_parquet(parquet_path)
        full_time = time.time() - tic
        print(f"all {full.shape[1]} columns:  {full.memory_usage(deep=True).sum() / 1e6:6.2f} MB in-memory,"
              f"  {full_time*1000:6.1f} ms to load")

        tic = time.time()
        subset = pd.read_parquet(parquet_path, columns=["MedInc", "MedHouseVal"])
        subset_time = time.time() - tic
        print(f"only 2 columns:   {subset.memory_usage(deep=True).sum() / 1e6:6.2f} MB in-memory,"
              f"  {subset_time*1000:6.1f} ms to load")

        subset.head()

**Expected output.** The projected read loads a fraction of the bytes and materialises a smaller frame in memory. In row-oriented CSV the same operation would still read the whole file — projection is only useful once the format itself is columnar.

### Choose the Right Data Types

Using appropriate data types reduces memory usage and speeds up computation:
- `int64` vs `int32` vs `int16` — most counters do not need 64 bits
- `float64` vs `float32` — half the memory for most ML features
- Categorical vs string for repeated values (`pd.Categorical` / `pl.Categorical`)
- Native date/datetime types vs string parsing at every access

In [ ]:
# Memory savings from right-sizing dtypes on California Housing.

# Add a low-cardinality string column to demonstrate the categorical case.
sample = df.copy()
sample["region"] = pd.cut(
    sample["Latitude"],
    bins=[-np.inf, 34, 36, 38, np.inf],
    labels=["SoCal", "Central-S", "Central-N", "NorCal"],
).astype(str)

before = sample.memory_usage(deep=True)
print("Before (default dtypes):")
print(before)
print(f"total: {before.sum() / 1e6:.2f} MB\n")

optimised = sample.copy()
float_cols = optimised.select_dtypes("float64").columns
optimised[float_cols] = optimised[float_cols].astype("float32")
optimised["region"] = optimised["region"].astype("category")

after = optimised.memory_usage(deep=True)
print("After (float32 + category):")
print(after)
print(f"total: {after.sum() / 1e6:.2f} MB  ({before.sum() / after.sum():.1f}x smaller)")

**Expected output.** Casting the float columns from `float64` to `float32` roughly halves the numeric memory footprint, and converting the low-cardinality string column to `category` collapses tens of thousands of Python string objects into a small integer array plus a dictionary of labels. Total memory typically drops by ~3-5x.

### Optimise Your Queries

Lazy evaluation enables query optimisation:
- Predicate pushdown (filter early)
- Projection pushdown (select columns early)
- Common subexpression elimination

In [ ]:
# Polars lazy evaluation: build a query plan, inspect it, then execute.
try:
    import polars as pl

    lazy = (
        pl.from_pandas(df)
        .lazy()
        .filter(pl.col("MedInc") > 5)
        .select(["MedInc", "AveRooms", "MedHouseVal"])
        .with_columns(pl.col("MedInc").round(0).alias("MedInc_rounded"))
        .group_by("MedInc_rounded")
        .agg(pl.mean("MedHouseVal").alias("mean_value"))
        .sort("MedInc_rounded")
    )

    print("Optimised query plan:\n")
    print(lazy.explain())
    print("\nResult:\n")
    print(lazy.collect())
except ImportError:
    print("Polars not installed. Install with: pip install polars")

### In-Memory vs On-Disk

| Approach | Pros | Cons |
|----------|------|------|
| In-memory (Polars/pandas) | Fast for small-medium data | Limited by RAM |
| On-disk (DuckDB/SQLite) | Handles data larger than RAM | Slightly more overhead |

### Relational Database Management Systems & SQL

Why databases {cite}`kleppmann2017designing`?
- **Data integrity** — constraints and transactions guarantee that concurrent writes leave the data in a consistent state
- **Concurrent access** — many readers and writers without race conditions
- **Query optimisation built-in** — the query planner picks indexes, join order, and execution strategy
- **Standardised query language (SQL)** — portable across engines

#### OLTP vs OLAP {cite}`kleppmann2017designing`

A single "database" abstraction hides two very different engineering targets. Kleppmann {cite}`kleppmann2017designing` draws the standard distinction:

| Property | **OLTP** (Online Transaction Processing) | **OLAP** (Online Analytical Processing) |
|---|---|---|
| Typical workload | Many small point reads/writes (fetch one customer, insert one order) | Few large scans (aggregate over millions of rows) |
| Row selectivity | Reads a handful of rows | Reads most rows, few columns |
| Storage layout | Row-oriented | Column-oriented |
| Latency target | Millisecond | Second to minute |
| Example engines | PostgreSQL, MySQL, SQLite | DuckDB, ClickHouse, BigQuery, Snowflake |

DuckDB is an *OLAP* engine — bulk reads, aggregations, wide tables. SQLite and PostgreSQL are *OLTP*-first. Modern warehouses (Snowflake, BigQuery) are OLAP built for the cloud. Choose the one matching your workload: a data-science project usually wants OLAP; an application backend usually wants OLTP.

In [ ]:
# DuckDB: query a pandas DataFrame directly, without loading it into a database first.
try:
    import duckdb

    result = duckdb.sql(
        """
        SELECT
            CASE
                WHEN MedInc < 2 THEN '0-2'
                WHEN MedInc < 4 THEN '2-4'
                WHEN MedInc < 6 THEN '4-6'
                WHEN MedInc < 8 THEN '6-8'
                ELSE '8+'
            END AS income_bin,
            COUNT(*)               AS n,
            AVG(MedHouseVal)       AS mean_value,
            AVG(AveRooms)          AS mean_rooms
        FROM df
        GROUP BY 1
        ORDER BY 1
        """
    ).df()
    print(result)
except ImportError:
    print("DuckDB not installed. Install with: pip install duckdb")

## Python and Databases

Key libraries for database interaction:
- **DuckDB** — embedded analytical database (OLAP), excellent for data science. Query pandas / Parquet / CSV directly with SQL.
- **SQLAlchemy** — ORM and database toolkit for production systems {cite}`huyen2022designing`. Speaks Postgres, MySQL, SQLite, and more through the same API.
- **sqlite3** — the built-in Python module. Perfect for prototypes, tests, and single-file storage.

In [ ]:
# SQLAlchemy: write a DataFrame to an in-memory SQLite database and query it back.
try:
    from sqlalchemy import create_engine, text

    engine = create_engine("sqlite:///:memory:")
    df[["MedInc", "AveRooms", "MedHouseVal"]].to_sql(
        "housing", engine, index=False, if_exists="replace"
    )

    with engine.connect() as conn:
        rows = conn.execute(
            text(
                """
                SELECT ROUND(MedInc) AS income, AVG(MedHouseVal) AS mean_value, COUNT(*) AS n
                FROM housing
                GROUP BY ROUND(MedInc)
                ORDER BY income
                LIMIT 5
                """
            )
        ).fetchall()
    for row in rows:
        print(row)
except ImportError:
    print("SQLAlchemy not installed. Install with: pip install sqlalchemy")

## Data Storage & Management Landscape

Overview of storage options:

| Format/System | Type | Best For |
|--------------|------|----------|
| CSV | File | Small data, interop |
| Parquet | File | Columnar analytics |
| SQLite | Embedded DB | Local apps, prototyping |
| DuckDB | Embedded DB | Analytics, data science |
| PostgreSQL | Server DB | Production systems |
| Delta Lake | Table format | Versioned data lakes |

## Repository Structure: .gitignore, Pre-commit, Type Hints

As our projects grow, we need better tooling:
- **`.gitignore`** — keep large data files, model artefacts, and secrets out of version control (see [Lecture 2](lecture_2.ipynb) for details)
- **Pre-commit hooks** — run formatters (`ruff format`), linters (`ruff check`), and type checkers (`mypy`) automatically on every commit
- **Type hints** — document expected types for better IDE support, self-documenting code, and static error checking

In [ ]:
# Type hints make function signatures precise and IDE-friendly.

def load_and_summarise(
    path: pathlib.Path,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    """Load a Parquet file (with optional column projection) and print a compact summary.

    Parameters
    ----------
    path
        Path to the Parquet file.
    columns
        Columns to read. ``None`` reads all columns.

    Returns
    -------
    pd.DataFrame
        The loaded (and possibly projected) frame.
    """
    frame = pd.read_parquet(path, columns=columns)
    print(f"loaded {frame.shape[0]:,} rows x {frame.shape[1]} cols from {path.name}")
    print(f"memory: {frame.memory_usage(deep=True).sum() / 1e6:.2f} MB")
    return frame


# Demonstrate on a synthetic frame we write to a temp file.
with tempfile.TemporaryDirectory() as tmp:
    demo_path = pathlib.Path(tmp) / "demo.parquet"
    try:
        df.to_parquet(demo_path, index=False)
        out = load_and_summarise(demo_path, columns=["MedInc", "MedHouseVal"])
        print(out.head())
    except ImportError:
        print("Install `pyarrow` to run the round-trip.")

## Scale Awareness: When Pandas Is Not Enough

### Row vs. columnar storage {cite}`kleppmann2017designing`

Every file format stores data as a sequence of *something*. The question is whether that something is a **row** (all columns of one observation together) or a **column** (all observations for one variable together).

| Format | Storage order | Read a full row | Read one column | Write one row |
|---|---|---|---|---|
| CSV | Row-oriented | Fast | Slow (scan all) | Append is easy |
| Parquet | Column-oriented | Slow (reconstruct) | **Very fast** | Requires rewrite |
| Arrow (in-memory) | Column-oriented | Slow | **Very fast** | Requires rewrite |

For analytics — aggregating, filtering, joining on a subset of columns — columnar storage is almost always faster and smaller *under typical analytical workloads* because:
1. Only the columns you touch are read from disk (**projection pushdown**)
2. Values within a column are similar, so compression ratios are much higher
3. SIMD vectorised CPU instructions operate on one column at a time

The same distinction underlies the split between **OLTP** engines (row-oriented; optimised for many small transactions) and **OLAP** engines (columnar; optimised for wide scans and aggregations) discussed by {cite}`kleppmann2017designing`.

:::{note}
A real-world example: a 1 GB CSV of 10M rows with 30 columns, where an analysis only uses 3 columns, requires reading all 1 GB from disk. The equivalent Parquet file may be 120 MB total and only ~15 MB of that needs to be read for those 3 columns. The exact figures vary with the codec, the query, and the hardware.
:::

### Lazy evaluation in Polars

Polars has two modes — **eager** (execute immediately) and **lazy** (build a query plan, optimise, then execute). The lazy API is what makes Polars competitive with on-disk databases for medium-sized data. `scan_parquet` + `collect` lets Polars decide how little of the file to read. For files larger than RAM this is not optional — it is the only way the query can complete.

The runnable Polars demo in section 2.4 shows the query plan Polars generates for a filter + project + group-by pipeline; on a Parquet source, `filter` and `select` are pushed down into the scan.

### SQL vs Python: where should the work happen? {cite}`kleppmann2017designing`

A recurring question is: given that both DuckDB/Postgres *and* pandas/Polars can filter, join, and aggregate, where should each step of a pipeline live?

A useful rule of thumb — consistent with the systems perspective in {cite}`kleppmann2017designing` — is:

> **Filter and aggregate in SQL; analyse in Python.**

The reasoning is mechanical, not aesthetic:

- **SQL engines have a query optimiser.** Given a declarative query, DuckDB or Postgres will pick the join order, push filters down to the scan, and use indexes or zone maps to skip data. A hand-written pandas pipeline does none of this automatically.
- **Data movement is the dominant cost.** Every row that crosses the process/network boundary from the database into Python is paid for in serialisation and memory. Filtering and aggregating server-side collapses billions of rows into thousands *before* they ever hit Python.
- **Python is unmatched for what comes next.** Once the data is small enough to fit comfortably in memory, scikit-learn, statsmodels, matplotlib, and PyTorch have no serious SQL equivalents.

The concrete workflow: push filters, joins, and group-by aggregations into SQL; return a compact result frame; do the modelling, plotting, and interpretation in Python.

### Knowing when you have outgrown pandas

| Signal | What it means | What to reach for |
|---|---|---|
| `MemoryError` on load | Data does not fit in RAM | Polars lazy / DuckDB |
| Load takes > 30 s | CSV at scale | Convert to Parquet first |
| A join on two large DataFrames is slow | pandas joins are single-threaded | Polars (multi-threaded) or DuckDB |
| Data is partitioned by date/region | Hive-partitioned Parquet | `scan_parquet(..., hive_partitioning=True)` |
| Data > 100 GB on a single machine | Single-node limit | Spark / distributed systems |

:::{tip}
The boundary is roughly: **< 1 GB — pandas is fine; 1–100 GB — Polars lazy or DuckDB; > 100 GB — Spark or a cloud data warehouse**.
The important skill is recognising which regime you are in *before* the compute bill or the memory error tells you {cite}`huyen2022designing`.
:::

(key-takeaways)=
## Key Takeaways

:::{admonition} Key Takeaways
:class: important
- Choose Parquet over CSV for any non-trivial dataset — smaller on disk, faster to load, cheaper to query column-wise.
- Right-size your data types (`float32`, `int32`, `category`) — memory savings compound into runtime savings.
- Use lazy evaluation (Polars `scan_*` + `collect`) so filters and projections are pushed down into the file read.
- Columnar formats (Parquet, Arrow) exist because analytics reads columns, not rows — OLAP and OLTP are different worlds {cite}`kleppmann2017designing`.
- Pick the right tool: pandas < 1 GB, Polars/DuckDB 1–100 GB, Spark or a warehouse > 100 GB.
- Databases give you integrity, concurrency, and a query optimiser you did not have to write.
- Keep large data out of Git; combine `.gitignore`, pre-commit, and type hints to keep production-ready code hygiene.
:::

(exercises)=
## Exercises

Work through these in a fresh notebook, using `load_california_housing()` as your starting frame.

**Exercise 1 — Dtype memory reduction.**
Load the California Housing data. Compute the total deep memory usage. Then:
1. Downcast every `float64` column to `float32`.
2. Bin `Latitude` into four labelled regions and store the result as a `category` dtype.
3. Report the memory usage before and after, and the compression ratio. What percentage of the saving comes from the categorical column alone?

**Exercise 2 — Query with DuckDB.**
Write the California Housing frame to a Parquet file in a temporary directory. Using DuckDB (`duckdb.sql(...)`), write a single query on the Parquet file (no intermediate pandas step) that returns, for each rounded `MedInc` bucket:
- the count of rows,
- the mean and median `MedHouseVal`,
- the mean `AveRooms`.
Order the result by income bucket. Time the query with `time.time()` and compare it to the equivalent pandas `groupby` on the in-memory frame.

**Exercise 3 — Parquet column projection.**
Save the California Housing data to two files: a CSV and a Parquet. Then measure the wall-clock time to read (a) all columns from CSV, (b) all columns from Parquet, (c) only `["MedInc", "MedHouseVal"]` from CSV (via `usecols=`), (d) only `["MedInc", "MedHouseVal"]` from Parquet (via `columns=`). Explain the ordering of the four numbers using what you learned about columnar storage.

:::{tip}
For all three exercises, wrap Polars/DuckDB imports in `try/except ImportError` so the notebook still runs on machines without those packages installed.
:::